# Bays & Brady (2024) Figure 5 — GP-Based Equivalent
## SD vs Set Size: Continuous Rise, No Plateau

### Experiment Family: `experiments.nature`
This is from the Bays, Schneegans & Brady (2024) *Nature Human Behaviour* paper
("Representation and Computation in Visual Working Memory"), **not** the Bays (2014)
*J. Neurosci.* paper. The two experiment families share the GP population coding
framework but test different theoretical claims and use independent utility functions.

---

### Scientific Question

The original Figure 5 from Bays & Brady (2024) showed a critical empirical finding:
**actual SD rises continuously with set size from 1 to 8 items, with no plateau.**

This was devastating for the slot model, which predicts that SD_normal (the precision
of "remembered" items) should plateau at small set sizes — the apparent continuous rise
being an artefact of mixing high-precision recalls with random guesses.

The variable-precision resource model explained the continuous rise by positing that
precision itself fluctuates from trial to trial. But the question remained: **does a
mechanistic neural model (not just a descriptive statistical model) reproduce this pattern?**

### What This Experiment Tests

We ask whether GP tuning curves + DN + Poisson noise produce:
- **(a)** Heavy-tailed pooled error distribution (aggregated across all set sizes)
- **(b)** Continuous rise in actual SD from 1–8 items, with no plateau
- **(c)** Agreement with the theoretical √l prediction from the DN activity cap

### The Theoretical Chain

The DN activity cap forces per-item gain = γN/l. This gives:
- Expected spikes per item = (γN/l) × T_d
- Fisher information ∝ spikes ∝ 1/l
- **Theoretical SD ∝ √l**

Deviations from √l reveal effects of GP heterogeneity (location-dependent
lengthscales) that go beyond the simple gain-scaling story. The GP model adds
heterogeneous tuning widths, which should produce MORE variable precision than
cosine tuning — potentially a better match to the continuous rise.

### Pipeline (per set size N, per trial)
1. Generate M neurons with GP tuning (lengthscale = λ_base)
2. DN with effective gain = γ/N (set size divides gain — the γ/N shortcut)
3. r_i(θ) = (γ/N) · g_i(θ) / (σ² + M⁻¹ Σ_j g_j(θ))
4. Spike counts: n_i ~ Poisson(r_i · T_d)
5. Full Poisson ML decode (rate-penalty retained): θ̂ = argmax Σ_i [n_i log g_i(θ) − g_i(θ)·T_d]
6. Circular error: ε = θ̂ − θ_true (wrapped to [-π, π))

**Note:** This uses the single-location decoder with the γ/N shortcut, matching
the design choice in the Bays (2014) figure_1 and figure_5 experiments. The γ/N
shortcut is valid because DN with N items sharing the population is mathematically
equivalent to single-item DN at reduced gain.

In [ ]:
# ============================================================
# CONFIGURATION — Modify these to explore
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import time

np.random.seed(42)
plt.rcParams.update({
    'font.family': 'sans-serif', 'font.size': 9,
    'axes.labelsize': 11, 'axes.titlesize': 12,
    'figure.dpi': 150,
})

# --- PARAMETERS (MODIFY THESE) ---
M = 100                    # Neurons per population
N_THETA = 64               # Orientation bins (resolution of θ grid)
N_TRIALS = 5000            # Trials per condition (10,000 in paper; 5,000 for speed)
T_D = 0.1                  # Decoding window (seconds)
SIGMA_SQ = 1e-6            # DN semi-saturation constant (≈0 for clean cap)
LAMBDA_BASE = 0.5          # GP lengthscale (controls tuning width)
GAMMA = 100.0              # Gain constant (Hz)
SET_SIZES = [1, 2, 3, 4, 5, 6, 7, 8]   # Full 1–8 range (key for seeing continuous rise)
SEED = 42                  # Master seed
N_SEEDS = 3                # Seeds for SE bands (5 in paper; 3 for speed)
N_BINS = 60                # Histogram bins for panel (a)

print(f"Config: M={M}, n_θ={N_THETA}, λ={LAMBDA_BASE}, γ={GAMMA} Hz")
print(f"Set sizes: {SET_SIZES}")
print(f"Per-item gain at N=1: {GAMMA/1:.0f} Hz, at N=8: {GAMMA/8:.1f} Hz")
print(f"Expected spikes at N=1: {GAMMA*T_D:.0f}, at N=8: {GAMMA*T_D/8:.1f}")

---
## Core Model

### What is occurring
We build the GP population generator, DN function, Poisson spike generator, and full
Poisson ML decoder inline. This makes the notebook entirely self-contained — no imports
from `core.encoder.*` or `core.decoder.*` needed.

The GP kernel is a periodic squared-exponential on the circle:
K(θ, θ') = exp(−½ · (|θ−θ'|_circ / λ)²)

Driving inputs are g_i(θ) = exp(f_i(θ)) where f_i ~ GP(0, K). The exponential
ensures positivity (firing rates must be non-negative).

In [ ]:
# ============================================================
# SELF-CONTAINED CORE MODEL
# ============================================================

def periodic_rbf_kernel(thetas, lengthscale):
    """Periodic squared-exponential kernel on the circle."""
    diff = thetas[:, None] - thetas[None, :]
    circ_dist = np.abs(np.angle(np.exp(1j * diff)))
    return np.exp(-0.5 * (circ_dist / lengthscale)**2)

def sample_gp_function(K, rng):
    """Sample one function from GP(0, K)."""
    L = np.linalg.cholesky(K + 1e-6 * np.eye(len(K)))
    return L @ rng.randn(len(K))

def generate_population(M, n_theta, lengthscale, seed):
    """
    Generate M neurons with GP tuning at a single lengthscale.
    Returns (thetas, g) where g = exp(f), shape (M, n_theta).
    """
    rng = np.random.RandomState(seed)
    thetas = np.linspace(-np.pi, np.pi, n_theta, endpoint=False)
    K = periodic_rbf_kernel(thetas, lengthscale)
    g = np.zeros((M, n_theta))
    for i in range(M):
        g[i] = np.exp(sample_gp_function(K, rng))
    return thetas, g

def dn_pointwise(r_pre, gamma, sigma_sq):
    """DN: r^post_i = γ · r^pre_i / (σ² + mean(r^pre))."""
    D = sigma_sq + np.mean(r_pre)
    return gamma * r_pre / D

def generate_spikes(rates, T_d, rng):
    """Poisson spike counts: n_i ~ Poisson(r_i · T_d)."""
    return rng.poisson(np.maximum(rates, 0) * T_d)

def compute_log_likelihood(counts, g, T_d):
    """
    Full Poisson ML log-likelihood (rate-penalty retained):
    LL(θ) = Σ_i [n_i · log g_i(θ) − g_i(θ) · T_d]

    This is the CORRECT decoder for GP tuning curves where
    Σ_i g_i(θ) is NOT constant across θ.
    """
    log_g = np.log(np.maximum(g, 1e-30))
    return counts @ log_g - T_d * np.sum(g, axis=0)

def compute_circular_error(theta_true, theta_hat):
    """Signed circular error in [-π, π)."""
    return np.angle(np.exp(1j * (theta_hat - theta_true)))

def circular_variance(errors):
    """Circular variance: σ² = −2·log(ρ₁) where ρ₁ = |mean(exp(iε))|."""
    m1 = np.mean(np.exp(1j * errors))
    rho1 = np.clip(np.abs(m1), 1e-15, 1.0 - 1e-10)
    return -2.0 * np.log(rho1)

def circular_sd_degrees(errors):
    """Circular SD in degrees."""
    return np.degrees(np.sqrt(circular_variance(errors)))

print("Core model loaded.")

---
## Main Experiment: Sweep Set Sizes 1–8

### What is occurring
For each seed (for SE estimation) and each set size N:
1. Generate a fresh GP population (same λ, different random seed)
2. Set effective gain = γ/N (the DN γ/N shortcut)
3. Run N_TRIALS: sample θ_true → DN at effective gain → Poisson → Full ML decode → error
4. Record circular SD in degrees

### Why the γ/N shortcut is valid
Under DN with N items sharing the population, total gain is divided by N items (each
item's representation competes for the activity budget). For the single-location decoder,
encoding one item at effective gain γ/N is mathematically equivalent to the full
multi-location encoding where the denominator grows by a factor of N.

### What to play with
- `GAMMA`: Higher → better precision at all set sizes (but √l scaling preserved)
- `LAMBDA_BASE`: Narrower tuning (smaller λ) → more Fisher info → lower absolute SD
- `SET_SIZES`: The key is [1,2,3,4,5,6,7,8] — all 8 points needed to see the continuous rise
- `N_SEEDS`: More seeds → tighter SE bands (5 is the paper standard; 3 for speed)

In [ ]:
# ============================================================
# MAIN EXPERIMENT
# ============================================================
t0 = time.time()
all_seeds = []

for s in range(N_SEEDS):
    current_seed = SEED + s * 1000
    thetas, g = generate_population(M, N_THETA, LAMBDA_BASE, current_seed)

    seed_data = {}
    for N in SET_SIZES:
        effective_gamma = GAMMA / N  # DN: gain divided by set size
        rng = np.random.RandomState(current_seed + N)

        errors = np.empty(N_TRIALS)
        for t in range(N_TRIALS):
            idx_true = rng.randint(N_THETA)

            # DN at true orientation with effective gain
            rates = dn_pointwise(g[:, idx_true], effective_gamma, SIGMA_SQ)

            # Poisson spikes
            counts = generate_spikes(rates, T_D, rng)

            # Full Poisson ML decode
            ll = compute_log_likelihood(counts, g, T_D)
            idx_hat = np.argmax(ll)

            errors[t] = compute_circular_error(thetas[idx_true], thetas[idx_hat])

        seed_data[N] = {
            'errors': errors,
            'sd_deg': circular_sd_degrees(errors),
            'variance': circular_variance(errors),
        }
        print(f"  seed={s} N={N}: SD = {seed_data[N]['sd_deg']:.2f}°")

    all_seeds.append(seed_data)

print(f"\nCompleted in {time.time()-t0:.1f}s")

---
## Aggregate Across Seeds and Compute Theoretical Prediction

### What is occurring
We compute mean ± SE of circular SD across seeds for each set size. We also pool
ALL errors across all set sizes and seeds into one distribution for panel (a).

The theoretical prediction is: SD(l) = SD(1) × √l, anchored at the empirical SD
at set size 1. Deviations from this line reveal effects that go beyond simple
gain scaling — for example, GP heterogeneity might cause slightly faster or slower
growth than √l.

In [ ]:
# ============================================================
# AGGREGATE AND THEORETICAL PREDICTION
# ============================================================

# Per-set-size statistics
summary = {}
for N in SET_SIZES:
    sds = [sd[N]['sd_deg'] for sd in all_seeds]
    vars_ = [sd[N]['variance'] for sd in all_seeds]
    summary[N] = {
        'sd_mean': np.mean(sds),
        'sd_se': np.std(sds, ddof=1) / np.sqrt(N_SEEDS) if N_SEEDS > 1 else 0,
        'variance_mean': np.mean(vars_),
        'variance_se': np.std(vars_, ddof=1) / np.sqrt(N_SEEDS) if N_SEEDS > 1 else 0,
    }

# Pooled errors across ALL set sizes and seeds (for panel a)
all_errors = []
for seed_data in all_seeds:
    for N in SET_SIZES:
        all_errors.append(seed_data[N]['errors'])
pooled_errors = np.concatenate(all_errors)

bin_edges = np.linspace(-np.pi, np.pi, N_BINS + 1)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
counts_hist, _ = np.histogram(pooled_errors, bins=bin_edges)
bin_width = bin_edges[1] - bin_edges[0]
pooled_density = counts_hist / (len(pooled_errors) * bin_width)
pooled_sd = circular_sd_degrees(pooled_errors)

# Theoretical √l prediction (anchored at N=1)
sd_at_1 = summary[SET_SIZES[0]]['sd_mean']
ns = np.array(SET_SIZES, dtype=float)
theoretical_sd = sd_at_1 * np.sqrt(ns)

print(f"Pooled actual SD = {pooled_sd:.1f}° ({len(pooled_errors):,} total errors)")
print(f"\nSD by set size:")
print(f"  {'N':<4} {'SD (°)':<10} {'Theory (°)':<12} {'Ratio':<8}")
for i, N in enumerate(SET_SIZES):
    s = summary[N]['sd_mean']
    t = theoretical_sd[i]
    print(f"  {N:<4} {s:<10.2f} {t:<12.2f} {s/t:<8.3f}")

---
## Three-Panel Figure

### Panel (a) — Pooled Error Distribution
All errors across all set sizes pooled into one histogram. The heavy tails are visible:
the distribution is NOT Gaussian (a Gaussian would drop off much faster). This heavy-tailed
shape is the pooled consequence of mixing narrow distributions (low set sizes) with broad
distributions (high set sizes).

### Panel (b) — Actual SD vs Set Size
The key result: SD rises **continuously** from 1 to 8 items. There is no plateau at
small set sizes (which would indicate a slot model). The error bars (SE across seeds)
show the estimate is stable.

### Panel (c) — Model vs Theoretical √l
The red line is the empirical SD; the blue line is the theoretical prediction SD ∝ √l
anchored at N=1. Close agreement means the DN activity cap is the dominant mechanism.
Deviations (especially at high N) may indicate GP heterogeneity effects or finite-population
corrections.

### What to play with
- Run with `SET_SIZES = [1, 2, 4, 8]` to see the coarser sampling used in Bays (2014)
- Compare `LAMBDA_BASE = 0.3` (sharp) vs `1.0` (broad) — absolute SD changes but √l shape preserved
- Try `GAMMA = 50` vs `200` — shifts all curves up/down but √l scaling remains

In [ ]:
# ============================================================
# THREE-PANEL FIGURE
# ============================================================
sd_mean = np.array([summary[N]['sd_mean'] for N in SET_SIZES])
sd_se = np.array([summary[N]['sd_se'] for N in SET_SIZES])

RED = '#CC2222'
BLACK = '#222222'
BLUE = '#2255AA'
GRAY_HIST = '#AAAAAA'

fig = plt.figure(figsize=(15, 5))
gs = gridspec.GridSpec(1, 3, width_ratios=[1.2, 1, 1],
                       wspace=0.35, left=0.06, right=0.97,
                       bottom=0.14, top=0.86)

# ── Panel (a): Pooled error distribution ──
ax_a = fig.add_subplot(gs[0])
ax_a.bar(np.degrees(bin_centers), pooled_density * np.pi / 180,
         width=np.degrees(bin_width) * 0.85,
         color=GRAY_HIST, edgecolor='#888888', linewidth=0.3,
         alpha=0.7, zorder=2)
ax_a.annotate(f'Actual SD = {pooled_sd:.1f}°',
              xy=(0.97, 0.95), xycoords='axes fraction',
              fontsize=10, ha='right', va='top',
              bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                        edgecolor='gray', alpha=0.9))
ax_a.set_xlim(-180, 180)
ax_a.set_xticks([-180, -90, 0, 90, 180])
ax_a.set_xlabel('Estimation error (degrees)')
ax_a.set_ylabel('Probability density')
ax_a.set_ylim(bottom=0)
ax_a.text(-0.14, 1.06, r'$\mathbf{a}$', transform=ax_a.transAxes,
          fontsize=15, fontweight='bold', va='top')

# ── Panel (b): Actual SD vs set size ──
ax_b = fig.add_subplot(gs[1])
ax_b.errorbar(ns, sd_mean, yerr=sd_se,
              fmt='o-', color=BLACK, linewidth=1.8, markersize=6,
              capsize=3, capthick=1.0, zorder=3, label='Actual SD')
ax_b.set_xlim(0.5, 8.5)
ax_b.set_xticks(SET_SIZES)
ax_b.set_ylim(bottom=0)
ax_b.set_xlabel('Set size')
ax_b.set_ylabel('SD (degrees)')
ax_b.set_title('GP Model', fontsize=11)
ax_b.legend(loc='upper left', fontsize=9, framealpha=0.9)
ax_b.text(-0.14, 1.06, r'$\mathbf{b}$', transform=ax_b.transAxes,
          fontsize=15, fontweight='bold', va='top')

# ── Panel (c): Model vs theoretical √l ──
ax_c = fig.add_subplot(gs[2])
ax_c.plot(ns, sd_mean, 'o-', color=RED, linewidth=1.8, markersize=6,
          zorder=3, label='GP model')
ax_c.fill_between(ns, sd_mean - sd_se, sd_mean + sd_se,
                   color=RED, alpha=0.15)
ax_c.plot(ns, sd_mean - sd_se, '--', color=RED, linewidth=0.6, alpha=0.5)
ax_c.plot(ns, sd_mean + sd_se, '--', color=RED, linewidth=0.6, alpha=0.5)

ns_smooth = np.linspace(1, 8, 100)
theoretical_smooth = sd_at_1 * np.sqrt(ns_smooth)
ax_c.plot(ns_smooth, theoretical_smooth, '-', color=BLUE, linewidth=1.8,
          alpha=0.7, zorder=2, label=r'Theoretical: SD $\propto \sqrt{l}$')

ax_c.set_xlim(0.5, 8.5)
ax_c.set_xticks(SET_SIZES)
ax_c.set_ylim(bottom=0)
ax_c.set_xlabel('Set size')
ax_c.set_ylabel('SD (degrees)')
ax_c.set_title('Model vs Theory', fontsize=11)
ax_c.legend(loc='upper left', fontsize=9, framealpha=0.9)
ax_c.text(-0.14, 1.06, r'$\mathbf{c}$', transform=ax_c.transAxes,
          fontsize=15, fontweight='bold', va='top')

fig.suptitle(
    f'GP Population Coding — Bays & Brady (2024) Fig 5 Equivalent '
    f'($\lambda$={LAMBDA_BASE}, $\gamma$={GAMMA} Hz, M={M})',
    fontsize=12, fontweight='bold', y=0.97)

plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()

---
## Interpretation

### What the continuous rise means
The slot model predicts that SD_normal (precision of "stored" items) should plateau
at around K=3-4 items — beyond that, extra items are simply not stored, so the
precision of stored items doesn't change. The continuous rise in actual SD without
any plateau is inconsistent with this prediction.

### What the √l agreement means
If the empirical SD tracks the theoretical √l curve closely, it means the DN
activity cap is the dominant mechanism: precision degrades because per-item gain
decreases as 1/l, giving Fisher information ∝ 1/l, giving SD ∝ √l.

### Where deviations from √l come from
The GP model with heterogeneous lengthscales (location-dependent λ) introduces
variable precision WITHIN the neural code — different neurons have different
effective tuning widths at different locations. This is a richer source of
precision variability than simple Poisson noise, and may cause deviations
from the pure √l prediction, especially at high set sizes where the
per-item spike count is very low and GP heterogeneity has a larger relative effect.

### The causal chain (complete)
1. GP tuning with heterogeneous lengthscales → variable precision across neurons and locations
2. DN caps total activity at γN → per-item gain = γN/l
3. Poisson spikes with expected count = (γN/l)·T_d → per-item SNR ∝ 1/√l
4. Full ML decoder → SD ∝ √l with deviations from GP heterogeneity
5. Pooling across set sizes → heavy-tailed distribution (mixture of narrow + broad)
6. **No plateau, continuous rise** — matching Bays & Brady (2024) empirical findings